# Business entity resolution - stage 1 (candidates) + stage 2 (matching)

**Run the notebook three times, top to bottom, changing only `STEP` in Stage 1 - Block 1:**

| `STEP` | what it does | output |
|---|---|---|
| `'learn'` | stage 1 on `N_LEARN` train S1: learns the dictionary, filler words, place aliases and the stage-1 ranker; stage-1 recall + missed pairs; a first stage-2 run | `work/stage1/learned/` |
| `'train'` | stage 1 on `N_TRAIN` other train S1 using what 'learn' saved (low RAM), then stage 2 trains the matching model with K-fold by S1 (= rotating 80/20 splits) and reports the honest F0.5 | `work/stage2/matcher_*` |
| `'test'` | stage 1 + stage 2 on the full test set, then checks the files against the challenge rules | `work/stage2/output/matching_results.tsv`, `candidate_pairs.tsv` |

**Stage 1:** cleaning (Indian scripts -> English with a dictionary learned from train pairs, legal words, filler words, number words -> digits, place aliases) -> hashed char 3-gram TF-IDF search from both sides (S1 -> S2/S3 penalised name / address searches; every S2/S3 record -> its best S1s, name-led and address-led) -> LightGBM ranker keeps ~9 candidates per S1.

**Stage 2:** stage-1 features + string similarities + how the records differ (house number cut off / one digit changed / different; typo-like vs different real words) -> LightGBM -> one owner per S2/S3 record -> threshold / expected-F0.5 rule chosen on cross-validated scores.

Every step prints its run time; the search cost grows with the number of S1, so time the 'learn' run first.

## Stage 1 - Block 1: settings, loading

In [ ]:
# ============================================================
# BLOCK 1 - setup, settings, loading data
# ============================================================
# !pip -q install anyascii sparse_dot_topn rapidfuzz pyarrow lightgbm   # <- needed in every new session
# from google.colab import drive; drive.mount('/content/drive')          # <- Colab only

import os, re, gc, time, json, collections, unicodedata, hashlib
import lightgbm as lgb                             # MIT licence
import numpy as np, pandas as pd, scipy.sparse as sp
from anyascii import anyascii                     # ISC licence, script -> Latin letters
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import HashingVectorizer
try:
    from sparse_dot_topn import sp_matmul_topn    # Apache-2.0, fast multi-threaded top-k product
except ImportError:
    sp_matmul_topn = None
    print("sparse_dot_topn not found -> slow scipy fallback")

# ---------------- paths
base_path = '/content/drive/My Drive/amazon-ml-26'
DATA_ROOT = f'{base_path}/datasets'   # folder with train/ and test/   Kaggle: '/kaggle/input/<your-dataset>/datasets'
WORK_ROOT = f'{base_path}/work'       # results are saved here           Kaggle: '/kaggle/working/work'
# ---------------- WHAT TO RUN (one notebook, three steps, run them in this order)
#  'learn': stage 1 on a train sample -> learns the dictionary, filler words, aliases and the
#           stage-1 ranker, prints stage-1 recall + missed pairs.            (stage 2 is skipped)
#  'train': stage 1 on a BIGGER train sample, using what 'learn' saved (fast, low RAM), then
#           stage 2 trains the matching model with K-fold cross-validation -> honest F0.5.
#  'test' : stage 1 + stage 2 on the whole test set -> output/matching_results.tsv + candidate_pairs.tsv
STEP      = 'learn'
N_LEARN   = 50_000     # S1 used by 'learn' (train split)
N_TRAIN   = 300_000    # S1 used by 'train' (never the 'learn' ones). None = all other train S1 (heavy)
SPLIT     = 'test' if STEP == 'test' else 'train'
LEARN     = STEP == 'learn'
DATA_DIR  = f'{DATA_ROOT}/{SPLIT}'
TRAIN_DIR = f'{DATA_ROOT}/train'                      # dictionary is always learned from train
WORK_DIR  = f'{WORK_ROOT}/stage1'
MODEL_DIR = f'{WORK_DIR}/learned'        # dictionary, filler words, aliases, ranker (made by SPLIT='train')
os.makedirs(MODEL_DIR, exist_ok=True)


# ---------------- how much to run
COUNTRIES     = None     # None = all countries, or e.g. ['India'] / ['US']
N_S1          = {'learn': N_LEARN, 'train': N_TRAIN, 'test': None}[STEP]   # S1 records searched
POOL_FRACTION = 1.0      # 1.0 = full S2/S3 pool -> honest recall. <1.0 = quick bug check only:
                         #   keeps the true matches + this share of the other records, so
                         #   recall looks BETTER than reality (fewer look-alikes).
SEED          = 42

# ---------------- candidate search settings
RETRIEVE_K     = 100     # S1 side: the penalised name search and address search each fetch this many
REV_K          = 10      # S2/S3 side: every S2/S3 record keeps its best REV_K S1 (reverse search),
                         #   ranked name-led AND address-led
REV_ADDR_FETCH = True    # S2/S3 side also FETCHES address-led (finds trade names at the right
                         #   address; roughly doubles the reverse-search time)
NAME_PENALTY   = 0.5     # name search score    = name - 0.5*(1 - address)  (no address: just name)
ADDR_PENALTY   = 0.25    # address search score = address - 0.25*(1 - name)
LEGAL_PENALTY  = 0.3     # minus this when legal forms really conflict (LLC vs Inc, Ltd vs LLP)
SEARCH_MAX_DF  = 5_000   # fast search skips query 3-grams found in more records than this...
KEEP_RAREST    = 8       # ...but every query always keeps its 8 rarest 3-grams
S1_CHUNK       = 20_000  # S1 records per chunk in the search (lower it if RAM runs out)

# ---------------- ranker settings (picks the final candidates from everything fetched)
P_MIN          = 0.001   # keep a candidate if the ranker's probability >= P_MIN ...
TOP_N          = 8       # ... or it is in the S1's top N by probability ...
KEEP_REV_BEST  = True    # ... or this S1 is the S2/S3 record's best S1, name-led or address-led
REV_BEST_CAP   = 20      #   (address-led: at most this many per S1, best addresses first)

N_THREADS      = os.cpu_count()
N_FEATURES     = 2 ** 22 # hashed token space

def tick(t0, msg):
    print(f"  {msg} ({time.time() - t0:.0f}s)", flush=True)

def read_source(path, countries=None, chunksize=1_000_000):
    """Read a TSV in chunks, optionally keeping only some countries (saves RAM)."""
    parts = []
    for ch in pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, chunksize=chunksize):
        if countries is not None:
            ch = ch[ch.country.str.strip().str.lower().isin([c.lower() for c in countries])]
        parts.append(ch)
    return pd.concat(parts, ignore_index=True)

def truth_pairs(gt):
    t = gt.assign(pool_id=gt.matched_entity_ids.str.split(',')).explode('pool_id')
    return t[t.pool_id != ''].rename(columns={'source1_entity_id': 's1_id'})[['s1_id', 'pool_id']] \
            .reset_index(drop=True)

t0 = time.time()
S1_ALL = read_source(f'{DATA_DIR}/{SPLIT}_source1.tsv', COUNTRIES)
POOL   = pd.concat([read_source(f'{DATA_DIR}/{SPLIT}_source2.tsv', COUNTRIES),
                    read_source(f'{DATA_DIR}/{SPLIT}_source3.tsv', COUNTRIES)], ignore_index=True)
tick(t0, f"loaded S1={len(S1_ALL):,} pool={len(POOL):,}")

rng = np.random.RandomState(SEED)
_pool_ids = S1_ALL.entity_id.values
if STEP == 'train' and os.path.exists(f"{MODEL_DIR}/learn_s1_ids.json"):   # never reuse 'learn' S1s
    _pool_ids = np.array(sorted(set(_pool_ids) - set(json.load(open(f"{MODEL_DIR}/learn_s1_ids.json")))))
if N_S1 is not None and N_S1 < len(_pool_ids):
    QUERY_IDS = set(_pool_ids[rng.choice(len(_pool_ids), N_S1, replace=False)])
else:
    QUERY_IDS = set(_pool_ids)
if STEP == 'learn':
    json.dump(sorted(QUERY_IDS), open(f"{MODEL_DIR}/learn_s1_ids.json", "w"))

TRUTH = None
if SPLIT == 'train':
    GT = pd.read_csv(f'{DATA_DIR}/train_ground_truth.tsv', sep='\t', dtype=str, keep_default_na=False)
    TRUTH_ALL = truth_pairs(GT)                              # every labelled pair (for the dictionary)
    TRUTH = TRUTH_ALL[TRUTH_ALL.s1_id.isin(QUERY_IDS)].reset_index(drop=True)
    print(f"  query S1: {len(QUERY_IDS):,} | their true pairs: {len(TRUTH):,} | "
          f"singletons: {len(QUERY_IDS - set(TRUTH.s1_id)):,}")

if POOL_FRACTION < 1.0:
    keep = rng.rand(len(POOL)) < POOL_FRACTION
    if TRUTH is not None:
        keep |= POOL.entity_id.isin(set(TRUTH.pool_id)).values
    POOL = POOL[keep].reset_index(drop=True)
    print(f"  DEBUG pool subset: {len(POOL):,} records (recall will look better than reality)")

## Stage 1 - Block 2: cleaning (dictionary, filler words, number words, aliases)

In [ ]:
# ============================================================
# BLOCK 2 - cleaning (+ learning a native-script -> English dictionary)
# ============================================================
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

# ---------------- country
COUNTRY_ALIASES = {"us": "us", "usa": "us", "u s": "us", "u s a": "us", "united states": "us",
                   "united states of america": "us", "india": "india", "in": "india", "ind": "india",
                   "bharat": "india", "france": "france", "fr": "france", "fra": "france"}

def clean_country(s):
    s = re.sub(r"\s+", " ", re.sub(r"[^a-z ]+", " ", anyascii(s or "").lower())).strip()
    return COUNTRY_ALIASES.get(s, s) or "unknown"   # unknown labels are kept as they are

# ---------------- native script handling
NATIVE_RE  = re.compile("[\u0370-\u1DFF\u1F00-\u1FFF\u2C00-\u2DFF\u2E80-\uFFEF]")   # non-Latin letters
_SEP       = r"\s\.,\-\(\)&/'\"#:;|_\[\]{}<>!?*+=~`@$%^" + "\u2019\u2018\u201c\u201d"
SPLIT_KEEP = re.compile(f"([{_SEP}]+)")
SPLIT_DROP = re.compile(f"[{_SEP}]+")

def _no_joiners(s):
    return s.replace("\u200c", "").replace("\u200d", "")

def native_pieces(s):
    return [p for p in SPLIT_DROP.split(_no_joiners(s)) if p]

def translit(s, D):
    """Replace native-script words with learned English words (fallback: anyascii later)."""
    if not D or not NATIVE_RE.search(s):
        return s
    return "".join(D.get(p, p) if NATIVE_RE.search(p) else p for p in SPLIT_KEEP.split(_no_joiners(s)))

# ---------------- names
LEGAL = {"inc", "incorporated", "llc", "llp", "lllp", "ltd", "limited", "pvt", "private", "corp",
         "corporation", "co", "company", "cos", "plc", "pc", "pllc", "lp", "pa", "dba", "opc",
         "sarl", "sas", "sasu", "eurl", "sci", "sa", "snc", "scop", "selarl", "scp", "cie",
         "praivet", "praibhet", "piraivet", "privte", "prvt", "praivrr", "pra", "li", "lim", "limitet",
         "limirrd", "elelpi"}
LEGAL_CANON = {"incorporated": "inc", "limited": "ltd", "limitet": "ltd", "li": "ltd", "lim": "ltd",
               "private": "pvt", "praivet": "pvt", "praibhet": "pvt", "piraivet": "pvt", "privte": "pvt",
               "prvt": "pvt", "praivrr": "pvt", "limirrd": "ltd", "pra": "pvt", "corporation": "corp",
               "company": "co", "cie": "co",
               "elelpi": "llp"}
LEET = str.maketrans({"1": "l", "0": "o", "5": "s", "3": "e", "4": "a"})
_url        = re.compile(r"https?://\S+|www\.")
_domain_end = re.compile(r"\.(com|c0m|net|org|co|in|fr|biz|info|us|io)\b(\.[a-z]{2})?")
_drop_char  = re.compile(r"['\u2019`.]")                 # "L.L.C." -> "llc", "Ph.D." -> "phd"
_elision    = re.compile(r"\b([a-z])['\u2019]")              # French "d'alsace" -> "d alsace" (addresses)
_amp        = re.compile(r"&|\+")
_non_word   = re.compile(r"[^a-z0-9 ]+")
_long_num   = re.compile(r"\b\d{5,}\b")                  # phone numbers / ids inside names
_spaces     = re.compile(r"\s+")
_letter_run = re.compile(r"\b(?:[a-z] ){1,}[a-z]\b")     # "l l c" -> "llc"
_dig_let    = re.compile(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)")   # "big93tattoo" -> "big 93 tattoo"
_lead_zero  = re.compile(r"\b0+(?=\d)")                  # "0684" -> "684" (numbers stay whole)

def _fix_leet(t):  # "tota1" -> "total"
    if t.isalpha() or t.isdigit():
        return t
    n = sum(c.isdigit() for c in t)
    return t.translate(LEET) if n == 1 and len(t) - n >= 3 else t

_MARKERS = {"dba", "fka", "aka", "formerly", "ta"}

def after_marker(toks):
    """'Avixylo dba Gonzalez Landscaping LLC' / 'Korlyra formerly: Carron Rising, LLC' ->
    keep the part AFTER the marker: that is the name the S1 record uses. A marker in first
    position ('DBA Organisers Pvt Ltd') is part of the real name and is left alone."""
    for k in range(1, len(toks)):
        t = toks[k]
        if t in _MARKERS or (t == "doing" and toks[k + 1:k + 3] == ["business", "as"]):
            rest = toks[k + (3 if t == "doing" else 1):]
            while rest and rest[0] in ("known", "as"):
                rest = rest[1:]
            if rest:
                return rest
    return toks

def clean_name(s, D=None):
    """-> (cleaned name, core name without legal words)"""
    s = translit(s or "", D)
    if not s:
        return "", ""
    if not s.isascii():
        s = anyascii(s)
    s = s.lower()
    s = _domain_end.sub(" ", _url.sub(" ", s))            # "foo-bar.com" -> "foo-bar"
    s = _amp.sub(" and ", _drop_char.sub("", s))
    s = _long_num.sub(" ", _non_word.sub(" ", s))
    toks = [_fix_leet(t) for t in s.split()]
    s = _dig_let.sub(" ", " ".join(toks))
    s = _letter_run.sub(lambda m: m.group(0).replace(" ", ""), _spaces.sub(" ", s).strip())
    s = _lead_zero.sub("", s)
    toks = after_marker([LEGAL_CANON.get(t, t) for t in s.split()])
    core = [t for t in toks if t not in LEGAL and t != "and"] or toks
    return " ".join(toks), " ".join(core)

# ---------------- addresses
US_STATES = {"alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca",
    "colorado": "co", "connecticut": "ct", "delaware": "de", "florida": "fl", "georgia": "ga",
    "hawaii": "hi", "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks",
    "kentucky": "ky", "louisiana": "la", "maine": "me", "maryland": "md", "massachusetts": "ma",
    "michigan": "mi", "minnesota": "mn", "mississippi": "ms", "missouri": "mo", "montana": "mt",
    "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj", "new mexico": "nm",
    "new york": "ny", "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
    "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc",
    "south dakota": "sd", "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt",
    "virginia": "va", "washington": "wa", "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy",
    "district of columbia": "dc"}
IN_STATES = {"andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as", "bihar": "br",
    "chhattisgarh": "cg", "goa": "ga", "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp",
    "jharkhand": "jh", "karnataka": "ka", "kerala": "kl", "keralam": "kl", "madhya pradesh": "mp",
    "maharashtra": "mh", "manipur": "mn", "meghalaya": "ml", "mizoram": "mz", "nagaland": "nl",
    "odisha": "od", "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk",
    "tamil nadu": "tn", "tamilnadu": "tn", "telangana": "ts", "tripura": "tr", "uttar pradesh": "up",
    "uttarakhand": "uk", "west bengal": "wb", "delhi": "dl", "jammu and kashmir": "jk",
    "chandigarh": "ch", "puducherry": "py", "pondicherry": "py", "ladakh": "la",
    # state names written in Indian scripts, as they look after anyascii
    "rajsthan": "rj", "uttr prdes": "up", "mharastr": "mh", "mhaaraassttr": "mh", "tmilnadu": "tn",
    "tmilnatu": "tn", "tmil natu": "tn", "krnatk": "ka", "krnattk": "ka", "kerl": "kl", "kerlm": "kl",
    "gujrat": "gj", "dilli": "dl", "pscim bmgal": "wb", "pscimbng": "wb", "pscim bng": "wb",
    "mdhy prdes": "mp", "hriyana": "hr", "pmjab": "pb", "amdhr prdes": "ap", "telmgana": "ts",
    "telmgan": "ts", "odisa": "od", "jharkhmd": "jh", "chttisgdh": "cg", "uttrakhmd": "uk",
    "himacl prdes": "hp", "gova": "ga"}
ADDR_WORDS = {"street": "st", "str": "st", "saint": "st", "avenue": "ave", "av": "ave", "avn": "ave",
    "road": "rd", "raod": "rd", "drive": "dr", "lane": "ln", "court": "ct", "boulevard": "blvd",
    "bd": "blvd", "bld": "blvd", "blv": "blvd", "place": "pl", "circle": "cir", "highway": "hwy",
    "parkway": "pkwy", "terrace": "ter", "trail": "trl", "square": "sq", "suite": "ste",
    "apartment": "apt", "floor": "fl", "building": "bldg", "north": "n", "south": "s", "east": "e",
    "west": "w", "mount": "mt", "fort": "ft", "township": "twp", "junction": "jn", "nagar": "ngr",
    "colony": "col", "sector": "sec",
    "r": "rue", "allee": "all", "chemin": "ch", "chem": "ch", "impasse": "imp", "route": "rte",
    "quai": "qu", "residence": "res"}                                   # last line: French
ADDR_STOP = {"null", "na", "nan", "none", "no", "near", "opp", "opposite", "po", "box", "dist",
             "district", "the", "of", "and", "de", "du", "des", "la", "le", "les", "d", "l"}

def _phrase_re(d):
    return re.compile(r"\b(" + "|".join(re.escape(k) for k in sorted(d, key=len, reverse=True)) + r")\b")
_us_re, _in_re = _phrase_re(US_STATES), _phrase_re(IN_STATES)

_UNITS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
          "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
          "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19}
_ORDS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5, "sixth": 6, "seventh": 7,
         "eighth": 8, "ninth": 9, "tenth": 10, "eleventh": 11, "twelfth": 12, "thirteenth": 13,
         "fourteenth": 14, "fifteenth": 15, "sixteenth": 16, "seventeenth": 17, "eighteenth": 18,
         "nineteenth": 19, "twentieth": 20, "thirtieth": 30, "fortieth": 40, "fiftieth": 50,
         "sixtieth": 60, "seventieth": 70, "eightieth": 80, "ninetieth": 90}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60, "seventy": 70,
         "eighty": 80, "ninety": 90}
_ord_end = re.compile(r"\b(\d+)(?:st|nd|rd|th)\b")      # "21st" -> "21", "45nd" -> "45"

def words_to_digits(toks):
    """'twenty first' -> '21', 'fourth' -> '4' ('21st' -> '21' is done before, by _ord_end),
    so '1001 Fourth Avenue' / '1001 4th Ave' and '135 Twenty First St' / '135 21st St' agree."""
    out, k = [], 0
    while k < len(toks):
        t = toks[k]
        nxt = toks[k + 1] if k + 1 < len(toks) else None
        if t in _TENS and nxt is not None and (0 < _UNITS.get(nxt, 0) < 10 or 0 < _ORDS.get(nxt, 0) < 10):
            out.append(str(_TENS[t] + _UNITS.get(nxt, _ORDS.get(nxt, 0)))); k += 2; continue
        if t in _TENS or t in _ORDS or t in _UNITS:
            out.append(str(_TENS.get(t, _ORDS.get(t, _UNITS.get(t)))))
        else:
            out.append(t)
        k += 1
    return out

_compound = re.compile(r"[a-z0-9]+(?:\s*[/-]\s*[a-z0-9]+)+")

def compound_numbers(s):
    """House numbers like '3/674 A', '6/A', 'TC 98/3715/4', 'AF-0684' are rare as a whole, but
    their parts ('3', '674') are common. Keep them as whole tokens too: '3_674', '98_3715',
    '98_3715_4' (every prefix, so a cut-off version still matches)."""
    out = []
    for m in _compound.finditer(s):
        parts = [p.lstrip("0") or "0" if p.isdigit() else p for p in re.split(r"\s*[/-]\s*", m.group(0))]
        if not any(ch.isdigit() for p in parts for ch in p):
            continue
        out += ["_".join(parts[:n]) for n in range(2, len(parts) + 1)]
    return out

def clean_address(s, D=None):
    s = translit(s or "", D)
    if not s:
        return ""
    if not s.isascii():
        s = anyascii(s)
    s = _elision.sub(r"\1 ", s.lower().replace("#", " "))
    comp = compound_numbers(s)
    s = _ord_end.sub(r"\1", s)
    s = _spaces.sub(" ", _non_word.sub(" ", _drop_char.sub("", s)))
    s = _in_re.sub(lambda m: IN_STATES[m.group(1)], _us_re.sub(lambda m: US_STATES[m.group(1)], s))
    s = _lead_zero.sub("", _dig_let.sub(" ", s))
    toks = [t for t in (ADDR_WORDS.get(t, t) for t in words_to_digits(s.split())) if t not in ADDR_STOP]
    return " ".join(toks + comp)

# ---------------- sound key (used to line up native words with English words)
_ph = [("ph", "f"), ("bh", "b"), ("dh", "d"), ("kh", "k"), ("gh", "g"), ("th", "t"),
       ("sh", "s"), ("ch", "c"), ("jh", "j"), ("ck", "k"), ("x", "ks")]
_skel_tr = str.maketrans({"c": "k", "q": "k", "g": "k", "j": "k", "z": "s", "w": "b",
                          "v": "b", "p": "b", "d": "t", "m": "n", "y": "a"})
def skeleton(w):
    for a, b in _ph:
        w = w.replace(a, b)
    return re.sub(r"(.)\1+", r"\1", re.sub(r"[aeiou]", "", w.translate(_skel_tr)))

def _roman_key(p):
    return skeleton(re.sub(r"[^a-z]", "", anyascii(p).lower()))

# ---------------- learn native word -> English word from labelled pairs
def learn_dictionary(pairs, max_pairs=300_000, min_count=2, min_share=0.6, seed=0):
    """pairs: DataFrame with s1_name, s1_addr (English side) and p_name, p_addr (pool side)."""
    has_nat = pairs.p_name.str.contains(NATIVE_RE) | pairs.p_addr.str.contains(NATIVE_RE)
    pairs = pairs[has_nat]
    if len(pairs) > max_pairs:
        pairs = pairs.sample(max_pairs, random_state=seed)
    cnt = collections.defaultdict(collections.Counter)

    def by_sound(nat, eng):
        keys = [skeleton(t) for t in eng]
        for p in nat:
            if NATIVE_RE.search(p):
                k = _roman_key(p)
                hit = process.extractOne(k, keys, scorer=fuzz.ratio, score_cutoff=60) if k else None
                if hit:
                    cnt[p][eng[hit[2]]] += 1

    for s1n, s1a, pn, pa in zip(pairs.s1_name, pairs.s1_addr, pairs.p_name, pairs.p_addr):
        if NATIVE_RE.search(pn):
            eng, nat = clean_name(s1n)[0].split(), native_pieces(pn)
            if len(eng) == len(nat):          # same word count -> line up by position
                for p, e in zip(nat, eng):
                    if NATIVE_RE.search(p):
                        cnt[p][e] += 1
            else:
                by_sound(nat, eng)
        if NATIVE_RE.search(pa):
            by_sound(native_pieces(pa), [t for t in clean_address(s1a).split()
                                         if not t.isdigit() and not t.startswith("#")])
    D = {}
    for p, c in cnt.items():
        (e, n), tot = c.most_common(1)[0], sum(c.values())
        if tot >= min_count and n / tot >= min_share:
            D[p] = e
    print(f"  dictionary: {len(D):,} native words learned from {len(pairs):,} pairs", flush=True)
    return D

# ---------------- clean whole frames (parallel)
_D_GLOBAL = None
def _clean_chunk(df):
    D = _D_GLOBAL
    names = [clean_name(x, D) for x in df.business_name]
    return pd.DataFrame({
        "entity_id": df.entity_id.values,
        "country":   [clean_country(x) for x in df.country],
        "name":      [n[0] for n in names],
        "name_core": [n[1] for n in names],
        "address":   [clean_address(x, D) for x in df.business_address],
        "nat_name":  df.business_name.str.contains(NATIVE_RE).values,
        "nat_addr":  df.business_address.str.contains(NATIVE_RE).values})

def clean_frame(df, D=None, workers=N_THREADS, n_chunks=None):
    global _D_GLOBAL
    _D_GLOBAL = D
    if len(df) == 0:
        return _clean_chunk(df)
    n_chunks = n_chunks or max(1, min(len(df) // 50_000, workers * 8))
    bounds = np.linspace(0, len(df), n_chunks + 1).astype(int)
    chunks = [df.iloc[a:b] for a, b in zip(bounds[:-1], bounds[1:])]
    if workers > 1 and len(df) > 100_000:
        # ProcessPoolExecutor stops with an error if a worker dies (e.g. out of RAM) instead of
        # waiting forever at 0% CPU like multiprocessing.Pool does
        with ProcessPoolExecutor(workers, mp_context=mp.get_context("fork")) as ex:
            out = list(ex.map(_clean_chunk, chunks))
    else:
        out = [_clean_chunk(c) for c in chunks]
    return pd.concat(out, ignore_index=True)

def cached_clean(df, tag, D):
    """Base cleaning (no dictionary) is cached on Drive; only native-script rows are re-cleaned
    with the dictionary, so re-runs take seconds instead of minutes."""
    ctag = "all" if COUNTRIES is None else "_".join(sorted(c.lower() for c in COUNTRIES))
    path = f"{WORK_DIR}/clean_v4_{SPLIT}_{tag}_{ctag}.parquet"   # v4 = current cleaning rules
    base = None
    if os.path.exists(path):
        base = pd.read_parquet(path)
        if len(base) != len(df) or not (base.entity_id.values == df.entity_id.values).all():
            base = None
    if base is None:
        base = clean_frame(df, None)
        base.to_parquet(path, index=False)
    nat = (base.nat_name | base.nat_addr).values
    if D and nat.any():
        fix = clean_frame(df[nat], D)
        for c in ["name", "name_core", "address"]:
            base.loc[nat, c] = fix[c].values
    return base

# ---------------- learned from train pairs: filler words and place-name aliases
TITLES = {"sri", "shri", "shree", "smt", "dr", "mr", "ms", "mrs", "mx"}

def learn_fillers(s1_core, p_core, p_native, min_count=50, min_share=0.4):
    """Words the noise ADDS to S2/S3 names ('center', 'services', 'the', 'fka', ...): a word is a
    filler when, in >= min_share of the S2/S3 names that contain it, the matching S1 name does not.
    Skipped: pairs with native-script names (a missing word there is a transliteration gap), and
    typo fragments like 'lndia' / 'roup' (they still carry signal through the 3-grams)."""
    added = collections.Counter()
    for a, b, nat in zip(s1_core, p_core, p_native):
        if not nat:
            added.update(set(b.split()) - set(a.split()))
    freq = collections.Counter(w for b in p_core for w in set(b.split()))
    s1f = collections.Counter(w for a in s1_core for w in set(a.split()))
    vocab = [w for w, n in s1f.items() if n >= 30] + sorted(LEGAL)
    F = set()
    for w, n in added.items():
        if n < min_count or n / freq[w] < min_share or w.isdigit() or len(w) < 2:
            continue
        typo = len(w) >= 4 and s1f[w] < 30 and process.extractOne(
            w, [v for v in vocab if v != w], scorer=fuzz.ratio, score_cutoff=80) is not None
        if not typo:
            F.add(w)
    return sorted(F | TITLES)

def drop_fillers(core, F):
    return " ".join(w for w in core.split() if w not in F) or core

def learn_aliases(s1_addr, p_addr, s1_ids, min_s1=10, min_lift=10.0, min_share=0.2, max_pairs=3_000_000, seed=0):
    """Place names used in place of each other ('maryvale' for 'phoenix', 'calicut' for
    'kozhikode'): for matching pairs, a word only in the S2/S3 address is paired with each word only
    in the S1 address. A pair is kept when it is seen for >= min_s1 different S1s and is >= min_lift
    times more common than chance (so generic pairs like 'door'/'hno' drop out). Look-alike
    spellings are skipped (the 3-grams handle typos). Returns {word: place word it stands for}."""
    idx = np.arange(len(s1_addr))
    if len(idx) > max_pairs:
        idx = np.random.RandomState(seed).choice(idx, max_pairs, replace=False)
    pair_s1 = collections.defaultdict(set); cx = collections.Counter(); cy = collections.Counter(); n = 0
    both = collections.Counter()
    for k in idx:
        A = {w for w in s1_addr[k].split() if len(w) >= 3 and w.isalpha()}
        B = {w for w in p_addr[k].split() if len(w) >= 3 and w.isalpha()}
        if not A or not B:
            continue
        n += 1
        ra, rb = A - B, B - A
        cx.update(ra); cy.update(rb); both.update(A & B)
        for x in ra:
            for y in rb:
                if fuzz.ratio(x, y) < 70:
                    pair_s1[(x, y)].add(s1_ids[k])
    best = {}
    for (x, y), ids in pair_s1.items():
        m = len(ids)
        lift = m * n / (cx[x] * cy[y])
        ok = m >= min_s1 and lift >= min_lift and m / cy[y] >= min_share and cy[y] >= both[y]
        if ok and (y not in best or lift > best[y][1]):
            best[y] = (x, lift)
    return {y: x for y, (x, _) in best.items()}

def add_aliases(addr, A):
    toks = set(addr.split())
    extra = sorted({A[w] for w in toks if w in A} - toks)
    return addr + (" " + " ".join(extra) if extra else "")

def self_pairs(s1c, pc):
    """Pairs that are almost surely the same business, found WITHOUT labels (used for countries
    that have no ground truth, e.g. France): same cleaned core name (>= 6 letters, unique among
    S1) and the same first house number."""
    first = lambda s: next((w for w in s.split() if w.isdigit()), None)
    a = pd.DataFrame({"k1": s1c.name_core.values, "k2": [first(x) for x in s1c.address], "i": np.arange(len(s1c))})
    b = pd.DataFrame({"k1": pc.name_core.values, "k2": [first(x) for x in pc.address], "j": np.arange(len(pc))})
    a = a[(a.k1.str.len() >= 6) & a.k2.notna()]; b = b[(b.k1.str.len() >= 6) & b.k2.notna()]
    a = a[~a.duplicated(["k1", "k2"], keep=False)]
    m = a.merge(b, on=["k1", "k2"])
    return m.i.values, m.j.values

# ---------------- run
t0 = time.time()
DICT_PATH = f"{MODEL_DIR}/dictionary.json"
if LEARN:
    exclude = QUERY_IDS if len(QUERY_IDS) < len(S1_ALL) else set()
    if not exclude:
        print("  WARNING: all S1 are queried, so everything is learned from the same pairs that "
              "are scored -> recall will be a bit optimistic.")
    # dictionary: only pool rows that contain native script are needed (keeps RAM low)
    pr = POOL[POOL.business_name.str.contains(NATIVE_RE) |
              POOL.business_address.str.contains(NATIVE_RE)].set_index("entity_id")
    tt = TRUTH_ALL[TRUTH_ALL.pool_id.isin(set(pr.index)) & ~TRUTH_ALL.s1_id.isin(exclude)]
    s1r = S1_ALL[S1_ALL.entity_id.isin(set(tt.s1_id))].set_index("entity_id")
    lp = pd.DataFrame({"s1_name": s1r.business_name.reindex(tt.s1_id).values,
                       "s1_addr": s1r.business_address.reindex(tt.s1_id).values,
                       "p_name": pr.business_name.reindex(tt.pool_id).values,
                       "p_addr": pr.business_address.reindex(tt.pool_id).values}).dropna()
    del pr, tt, s1r; gc.collect()
    D = learn_dictionary(lp)
    json.dump(D, open(DICT_PATH, "w", encoding="utf-8"), ensure_ascii=False)
else:
    D = json.load(open(DICT_PATH, encoding="utf-8"))
    print(f"  dictionary loaded: {len(D):,} words")
tick(t0, "dictionary ready")

t0 = time.time()
S1_CLEAN   = cached_clean(S1_ALL, "s1", D)
POOL_CLEAN = cached_clean(POOL, "pool", D) if POOL_FRACTION == 1.0 else clean_frame(POOL, D)
tick(t0, "cleaned")

# filler words and place-name aliases: learned on train (never from the queried S1s), saved,
# and loaded for test. Countries without ground truth (France) get aliases from label-free pairs.
t0 = time.time()
FILL_PATH, ALIAS_PATH = f"{MODEL_DIR}/filler_words.json", f"{MODEL_DIR}/aliases.json"
if LEARN:
    s1pos = pd.Series(np.arange(len(S1_CLEAN)), index=S1_CLEAN.entity_id.values)
    ppos = pd.Series(np.arange(len(POOL_CLEAN)), index=POOL_CLEAN.entity_id.values)
    lt = TRUTH_ALL[~TRUTH_ALL.s1_id.isin(exclude) & TRUTH_ALL.pool_id.isin(ppos.index) &
                   TRUTH_ALL.s1_id.isin(s1pos.index)]
    li, lj = s1pos.reindex(lt.s1_id).values, ppos.reindex(lt.pool_id).values
    FILLERS = learn_fillers(S1_CLEAN.name_core.values[li], POOL_CLEAN.name_core.values[lj],
                            POOL_CLEAN.nat_name.values[lj])
    ALIASES = {}
    for country in sorted(S1_CLEAN.country.unique()):
        m = S1_CLEAN.country.values[li] == country
        ALIASES[country] = learn_aliases(S1_CLEAN.address.values[li[m]], POOL_CLEAN.address.values[lj[m]],
                                         lt.s1_id.values[m])
    json.dump(FILLERS, open(FILL_PATH, "w")); json.dump(ALIASES, open(ALIAS_PATH, "w"))
else:
    FILLERS, ALIASES = json.load(open(FILL_PATH)), json.load(open(ALIAS_PATH))
FSET = set(FILLERS)
S1_CLEAN["name_core"] = [drop_fillers(x, FSET) for x in S1_CLEAN.name_core]
POOL_CLEAN["name_core"] = [drop_fillers(x, FSET) for x in POOL_CLEAN.name_core]
for country in sorted(S1_CLEAN.country.unique()):
    if country not in ALIASES:        # no ground truth for this country -> label-free pairs
        s1c = S1_CLEAN[S1_CLEAN.country == country]; pc = POOL_CLEAN[POOL_CLEAN.country == country]
        i, j = self_pairs(s1c, pc)
        ALIASES[country] = learn_aliases(s1c.address.values[i], pc.address.values[j], s1c.entity_id.values[i])
        print(f"  {country}: {len(i):,} label-free pairs -> {len(ALIASES[country]):,} alias words")
    for df in (S1_CLEAN, POOL_CLEAN):
        m = (df.country == country).values
        if ALIASES[country] and m.any():
            df.loc[m, "address"] = [add_aliases(x, ALIASES[country]) for x in df.address.values[m]]
tick(t0, f"{len(FILLERS)} filler words, aliases per country: " +
     ", ".join(f"{k}={len(v)}" for k, v in ALIASES.items()))
print("  filler words:", ", ".join(FILLERS[:40]))
for k, v in ALIASES.items():
    print(f"  {k} alias examples:", ", ".join(f"{w}->{x}" for w, x in list(v.items())[:12]))

# a few native-script examples, before -> after
ex = POOL[POOL.business_name.str.contains(NATIVE_RE)].head(6).index
if len(ex):
    print(pd.DataFrame({"raw": POOL.business_name[ex].values, "clean": POOL_CLEAN.name[ex].values,
                        "anyascii only": [clean_name(x)[0] for x in POOL.business_name[ex]]}).to_string(index=False))

## Stage 1 - Block 3: candidate search from both sides + features

In [ ]:
# ============================================================
# BLOCK 3 - candidate search (hashed char 3-gram TF-IDF) + features, per country
# ============================================================
# For every S1 we FETCH a generous set of pool records from two directions:
#   S1 side  : penalised name search and penalised address search (RETRIEVE_K each)
#   S2/S3 side: every S2/S3 record searches all S1 and keeps its best REV_K (reverse search).
#              S1 has no duplicates, so this side is not crowded by copies of look-alikes.
# Each fetched pair gets cheap features (exact TF-IDF scores, penalised scores, ranks from both
# sides, house-number and word overlap, legal-form conflict). BLOCK 3b trains a small ranker on
# them (train) or applies it (test) to keep only the final few candidates per S1.
# Each record is turned into a sparse TF-IDF vector ONCE. For L2-normalised vectors
# cosine = dot product, so "one S1 vs every pool record" is one row of the sparse product
# S1 @ Pool^T. A sparse product only visits pairs that share at least one 3-gram, and
# sparse_dot_topn keeps only the top-k per row while multiplying (multi-threaded C++).
#
# Two penalised searches (one shared index = name vectors + address vectors side by side):
#   name search    : name - NAME_PENALTY*(1 - address)   (S2/S3 with no address: just name,
#                    so a missing address is neither rewarded nor punished)
#   address search : address - ADDR_PENALTY*(1 - name)   (light, so trade names at the right
#                    address still get in)
# Same name + different address sinks; ties between identical names are broken by address.
# A real legal-form conflict (LLC vs Inc, Ltd vs LLP) subtracts LEGAL_PENALTY afterwards.
#
# Speed trick (query pruning): very common 3-grams ("ent", " ro") have huge posting lists and
# make the product slow, so the fast search skips query 3-grams seen in more than SEARCH_MAX_DF
# records - except that each query keeps its KEEP_RAREST rarest 3-grams. The pool index keeps
# everything, so every pool record is compared on the same 3-grams (fair ranking). The fetched
# candidates are then re-scored with the EXACT cosine (all 3-grams), and the threshold and
# top-K are applied on that exact score.

def name_grams(core):
    """Name without spaces -> 3-grams with start/end marks; numbers stay whole.
    'maure williams colombier' and 'maurewilliamscolombier' give almost the same grams."""
    letters, nums = [], []
    for w in core.split():
        (nums if w.isdigit() else letters).append(w)
    s = "^" + "".join(letters) + "$"
    return [s[i:i + 3] for i in range(len(s) - 2)] + ["#" + n for n in nums]

def addr_grams(addr):
    """3-grams inside each address word (order of words does not matter); numbers stay whole."""
    out = []
    for w in addr.split():
        if w.isdigit() or "_" in w:          # numbers and compound house numbers stay whole
            out.append("#" + w)
        else:
            s = " " + w + " "
            out.extend(s[i:i + 3] for i in range(len(s) - 2))
    return out

_HV = None
def _hash_chunk(texts):
    return _HV.transform(texts)

def hash_texts(hv, texts, chunk=200_000):
    """HashingVectorizer.transform in parallel chunks (it is pure Python per record)."""
    global _HV
    _HV = hv
    chunks = [texts[i:i + chunk] for i in range(0, len(texts), chunk)]
    if not chunks:
        return sp.csr_matrix((0, N_FEATURES), dtype=np.float32)
    if N_THREADS > 1 and len(chunks) > 1:
        with ProcessPoolExecutor(N_THREADS, mp_context=mp.get_context("fork")) as ex:   # errors, never hangs
            mats = list(ex.map(_hash_chunk, chunks))
    else:
        mats = [_hash_chunk(c) for c in chunks]
    return sp.vstack(mats, format="csr")

def rowwise_dot(A, B, i, j, chunk=500_000):
    out = np.empty(len(i), np.float32)
    for st in range(0, len(i), chunk):
        out[st:st + chunk] = np.asarray(A[i[st:st + chunk]].multiply(B[j[st:st + chunk]]).sum(1)).ravel()
    return out

class Space:
    """TF-IDF space of one field (name or address) for one country.
    A = exact vectors of ALL S1 of the country, B = exact vectors of the pool."""
    def __init__(self, analyzer, s1_text, pool_text):
        self.hv = HashingVectorizer(analyzer=analyzer, n_features=N_FEATURES, alternate_sign=False,
                                    norm=None, binary=True, dtype=np.float32)
        Ms, Mp = hash_texts(self.hv, s1_text), hash_texts(self.hv, pool_text)
        df = (np.bincount(Ms.indices, minlength=N_FEATURES) +
              np.bincount(Mp.indices, minlength=N_FEATURES)).astype(np.int32)
        n = Ms.shape[0] + Mp.shape[0]
        self.df = df
        self.idf = (np.log((1 + n) / (1 + df)) + 1).astype(np.float32)
        self.idf[df < 2] = 0                           # a 3-gram seen once can never be shared
        self.A, self.B = self._weight(Ms), self._weight(Mp)
        del Ms, Mp

    def _weight(self, M):
        M = M.copy()
        M.data = self.idf[M.indices]
        M.eliminate_zeros()
        norm = np.sqrt(np.asarray(M.multiply(M).sum(1)).ravel()); norm[norm == 0] = 1
        return (sp.diags((1 / norm).astype(np.float32)) @ M).tocsr()

    def prune(self, M):
        """fast-search version of query vectors: drop very common 3-grams, keep the rarest few"""
        M = M.copy()
        common = self.df[M.indices] > SEARCH_MAX_DF
        if common.any():
            rows = np.repeat(np.arange(M.shape[0]), np.diff(M.indptr))
            order = np.lexsort((-M.data, rows))
            rank = np.empty(len(order), np.int32)
            rank[order] = np.arange(len(order)) - M.indptr[rows[order]]
            M.data[common & (rank >= KEEP_RAREST)] = 0
            M.eliminate_zeros()
        return M

def top_k(Q, index, k, batch=20_000, transposed=False):
    """For each row of Q: up to k rows of `index` with the highest dot product (> 0).
    transposed=True: `index` is already (features x records)."""
    It = index if transposed else index.T.tocsr()
    R, C, V = [], [], []
    for st in range(0, Q.shape[0], batch):
        q = Q[st:st + batch]
        if sp_matmul_topn is not None:
            M = sp_matmul_topn(q, It, top_n=k, threshold=0.0, n_threads=N_THREADS).tocoo()
            r, c, v = M.row, M.col, M.data
        else:
            M = (q @ It).tocoo()
            o = np.lexsort((-M.data, M.row))
            r, c, v = M.row[o], M.col[o], M.data[o]
            keep = (np.arange(len(r)) - np.searchsorted(r, r)) < k
            r, c, v = r[keep], c[keep], v[keep]
        R.append(r.astype(np.int64) + st); C.append(c.astype(np.int64)); V.append(v)
    if not transposed:
        del It
    return np.concatenate(R), np.concatenate(C), np.concatenate(V)

LEGAL_FORM_WORDS = {"inc", "llc", "llp", "lllp", "ltd", "pvt", "corp", "co", "cos", "plc", "pc", "pllc",
                    "lp", "pa", "opc", "sarl", "sas", "sasu", "eurl", "sci", "sa", "snc"}

def legal_form(name):
    return " ".join(sorted({t for t in name.split() if t in LEGAL_FORM_WORDS}))

def conflict_table(forms):
    """codes + table[c1, c2] = True when both have a legal form and neither contains the other
    (LLC vs Inc). 'pvt ltd' vs 'pvt' is NOT a conflict: the noise often cuts the last word."""
    codes, uniq = pd.factorize(forms)
    sets = [frozenset(u.split()) for u in uniq]
    table = np.array([[bool(a) and bool(b) and not (a <= b or b <= a) for b in sets] for a in sets])
    return codes, table

def rank_in_group(g, v):
    """rank (1 = best) of value v inside each group g"""
    o = np.lexsort((-v, g))
    r = np.empty(len(g), np.int32)
    r[o] = np.arange(len(g)) - np.searchsorted(g[o], g[o]) + 1
    return r

def _words(s):
    return s.split()

def _numbers(s):
    return [w for w in s.split() if w.isdigit() or "_" in w]

def binary_matrix(texts, analyzer):
    hv = HashingVectorizer(analyzer=analyzer, n_features=2 ** 20, alternate_sign=False, norm=None,
                           binary=True, dtype=np.float32)
    return hash_texts(hv, list(texts)), hv

FEATS = ["n", "a", "empty", "legal", "pen_n", "pen_a", "r_rank", "pool_best", "margin", "rank_n", "rank_a",
         "ra_rank", "pool_best_a", "margin_a",
         "num_j", "num_first", "num_sub", "num_extra", "w_share_p", "w_share_s", "first_eq", "nw_s", "nw_p"]
MONO = {"n": 1, "a": 1, "pen_n": 1, "pen_a": 1, "num_j": 1, "num_first": 1, "num_sub": 1, "w_share_p": 1,
        "w_share_s": 1, "first_eq": 1, "legal": -1, "r_rank": -1, "margin": -1, "rank_n": -1, "rank_a": -1,
        "num_extra": -1, "ra_rank": -1, "margin_a": -1}   # "more similar can never mean less likely" -> no crazy scores

def pair_features(i, j, r_rank, ra_rank, X):
    """features for pairs (S1 row i, pool row j) of one country; X = per-country context"""
    lam, mu = NAME_PENALTY, ADDR_PENALTY
    f = {"n": rowwise_dot(X["nm"].A, X["nm"].B, i, j), "a": rowwise_dot(X["ad"].A, X["ad"].B, i, j)}
    f["empty"] = X["empty"][j].astype(np.float32)
    f["legal"] = X["ctab"][X["c1"][i], X["c2"][j]].astype(np.float32)
    lp = LEGAL_PENALTY * f["legal"]
    f["pen_n"] = np.where(X["empty"][j], f["n"], f["n"] - lam * (1 - f["a"])) - lp
    f["pen_a"] = f["a"] - mu * (1 - f["n"]) - lp
    f["r_rank"] = r_rank.astype(np.float32)
    f["pool_best"] = X["pool_best"][j]
    f["margin"] = f["pool_best"] - f["pen_n"]
    f["ra_rank"] = ra_rank.astype(np.float32)                 # address-led reverse rank
    f["pool_best_a"] = X["pool_best_a"][j]
    f["margin_a"] = f["pool_best_a"] - f["pen_a"]
    f["rank_n"] = rank_in_group(i, f["pen_n"]).astype(np.float32)
    f["rank_a"] = rank_in_group(i, f["pen_a"]).astype(np.float32)
    inter = rowwise_dot(X["NS"], X["NP"], i, j)
    cs, cp = X["ns_cnt"][i], X["np_cnt"][j]
    f["num_j"] = np.where((cs > 0) & (cp > 0), inter / np.maximum(cs + cp - inter, 1), -1).astype(np.float32)
    f["num_sub"] = ((cp > 0) & (inter >= cp)).astype(np.float32)
    f["num_extra"] = (cp - inter).astype(np.float32)
    col = X["first_num"][i]; ok = col >= 0
    fn = np.zeros(len(i), np.float32)
    if ok.any():
        fn[ok] = np.asarray(X["NP"][j[ok], col[ok]]).ravel()
    f["num_first"] = np.where(ok & (cp > 0), fn, -1).astype(np.float32)
    w = rowwise_dot(X["WS"], X["WP"], i, j)
    f["nw_s"], f["nw_p"] = X["ws_cnt"][i], X["wp_cnt"][j]
    f["w_share_p"] = (w / np.maximum(f["nw_p"], 1)).astype(np.float32)
    f["w_share_s"] = (w / np.maximum(f["nw_s"], 1)).astype(np.float32)
    f["first_eq"] = ((X["fw_s"][i] == X["fw_p"][j]) & (X["fw_s"][i] >= 0)).astype(np.float32)
    return pd.DataFrame(f)

def select(df):
    """final candidates: ranker p >= P_MIN, or the S1's top TOP_N, or the S2/S3 record's best S1
    (name-led, and address-led capped at REV_BEST_CAP per S1: an S1 whose address is only a city,
    e.g. 'Ahmedabad, Gujarat', is the address-led best of thousands of records)"""
    keep = np.array(((df.p >= P_MIN) | (df.rank_p <= TOP_N)).values, dtype=bool)   # writable copy
    if KEEP_REV_BEST:
        keep |= (df.r_rank == 1).values
        extra = (df.ra_rank == 1).values & ~keep
        if extra.any():
            sub = df[extra].sort_values(["s1_id", "a"], ascending=[True, False])
            ok = (sub.groupby("s1_id").cumcount() < REV_BEST_CAP).values
            keep[df.index.get_indexer(sub.index[ok])] = True
    return keep

RANKER = None if LEARN else lgb.Booster(model_file=f"{MODEL_DIR}/ranker.txt")   # 'train'/'test': apply it
FETCHED, CANDS, TRUE_SCORES = [], [], []
T_ALL = time.time()
# checkpoints: each finished country is saved; after a crash / restart it is loaded instead of redone.
# The name contains a fingerprint of the settings, so changing a setting starts fresh automatically.
_sig = json.dumps([STEP, N_S1, SEED, POOL_FRACTION, COUNTRIES, RETRIEVE_K, REV_K, REV_ADDR_FETCH, SEARCH_MAX_DF,
                   KEEP_RAREST, NAME_PENALTY, ADDR_PENALTY, LEGAL_PENALTY, P_MIN, TOP_N, KEEP_REV_BEST,
                   REV_BEST_CAP], default=str)
CKPT_TAG = f"{SPLIT}_n{N_S1}_{hashlib.md5(_sig.encode()).hexdigest()[:8]}"
for country in sorted(S1_CLEAN.country.unique()):
    s1c = S1_CLEAN[S1_CLEAN.country == country].reset_index(drop=True)
    pc = POOL_CLEAN[POOL_CLEAN.country == country].reset_index(drop=True)
    qrows = np.where(s1c.entity_id.isin(QUERY_IDS).values)[0]
    if len(qrows) == 0 or len(pc) == 0:
        continue
    ckpt = f"{WORK_DIR}/ckpt_{CKPT_TAG}_{country}"
    if os.path.exists(ckpt + "_fetched.parquet"):
        FETCHED.append(pd.read_parquet(ckpt + "_fetched.parquet"))
        if os.path.exists(ckpt + "_true.parquet"):
            TRUE_SCORES.append(pd.read_parquet(ckpt + "_true.parquet"))
        print(f"{country}: loaded from checkpoint {os.path.basename(ckpt)}_*.parquet", flush=True)
        continue
    n_before = len(FETCHED)
    lam, mu = NAME_PENALTY, ADDR_PENALTY
    print(f"{country}: query S1={len(qrows):,} (of {len(s1c):,}) | pool={len(pc):,}", flush=True)
    t0 = time.time()
    X = {"nm": Space(name_grams, s1c.name_core.tolist(), pc.name_core.tolist()),
         "ad": Space(addr_grams, s1c.address.tolist(), pc.address.tolist()),
         "empty": pc.address.eq("").values}
    codes, X["ctab"] = conflict_table(np.concatenate([[legal_form(x) for x in s1c.name.values],
                                                      [legal_form(x) for x in pc.name.values]]))
    X["c1"], X["c2"] = codes[:len(s1c)], codes[len(s1c):]
    X["NS"], hvn = binary_matrix(s1c.address.values, _numbers); X["NP"], _ = binary_matrix(pc.address.values, _numbers)
    X["WS"], _ = binary_matrix(s1c.name_core.values, _words); X["WP"], _ = binary_matrix(pc.name_core.values, _words)
    X["ns_cnt"] = np.diff(X["NS"].indptr).astype(np.float32); X["np_cnt"] = np.diff(X["NP"].indptr).astype(np.float32)
    X["ws_cnt"] = np.diff(X["WS"].indptr).astype(np.float32); X["wp_cnt"] = np.diff(X["WP"].indptr).astype(np.float32)
    fnum = [next((w for w in s.split() if w.isdigit()), None) for s in s1c.address.values]
    FN = hash_texts(hvn, [x or "" for x in fnum])            # first house number of each S1
    X["first_num"] = np.full(len(s1c), -1, np.int64)
    has = np.diff(FN.indptr) > 0
    X["first_num"][has] = FN.indices[FN.indptr[:-1][has]]
    fw = pd.factorize(np.concatenate([[x.split()[0] if x else "" for x in s1c.name_core.values],
                                      [x.split()[0] if x else "" for x in pc.name_core.values]]))[0]
    fw[np.concatenate([s1c.name_core.eq("").values, pc.name_core.eq("").values])] = -1
    X["fw_s"], X["fw_p"] = fw[:len(s1c)], fw[len(s1c):]
    tick(t0, "vectors built")

    # ---- S2/S3 side: every pool record ranks S1s two ways and keeps its best REV_K of each:
    #   name-led   : name - lam*(1 - address)  (no address: just name)
    #   address-led: address - mu*(1 - name)   (finds trade names like "Drexkor" at the right address)
    # Both rankings are computed over the union of a name-led fetch and (REV_ADDR_FETCH) an
    # address-led fetch, so each side can bring in S1s the other would never look at.
    t0 = time.time()
    It1 = sp.vstack([X["nm"].A.T, X["ad"].A.T * np.float32(lam)], format="csr")
    It1a = sp.vstack([X["nm"].A.T * np.float32(mu), X["ad"].A.T], format="csr") if REV_ADDR_FETCH else None
    isq = np.zeros(len(s1c), bool); isq[qrows] = True
    X["pool_best"] = np.full(len(pc), -1.0, np.float32)
    X["pool_best_a"] = np.full(len(pc), -1.0, np.float32)
    ns1, cap = len(s1c), REV_K + 1
    rv = []
    for st in range(0, len(pc), 50_000):
        Bn, Ba = X["nm"].B[st:st + 50_000], X["ad"].B[st:st + 50_000]
        Q = sp.hstack([X["nm"].prune(Bn), X["ad"].prune(Ba)], format="csr")
        pj, si, _ = top_k(Q, It1, 2 * REV_K, transposed=True)
        if It1a is not None:
            has_a = ~X["empty"][st:st + 50_000]
            pj2, si2, _ = top_k(Q[has_a], It1a, 2 * REV_K, transposed=True)
            pj2 = np.where(has_a)[0][pj2]
            key = np.unique(np.concatenate([pj * ns1 + si, pj2 * ns1 + si2]))
            pj, si = key // ns1, key % ns1
        n = rowwise_dot(X["nm"].A, Bn, si, pj); a = rowwise_dot(X["ad"].A, Ba, si, pj)
        e = X["empty"][pj + st]
        lp = LEGAL_PENALTY * X["ctab"][X["c1"][si], X["c2"][pj + st]]
        pen = np.where(e, n, n - lam * (1 - a)) - lp
        pen_a = np.where(e, -9.0, a - mu * (1 - n) - lp)          # no address -> no address-led rank
        r, ra = rank_in_group(pj, pen), rank_in_group(pj, pen_a)
        ra[e] = cap
        b1 = r == 1; X["pool_best"][pj[b1] + st] = pen[b1]
        b2 = (ra == 1) & ~e; X["pool_best_a"][pj[b2] + st] = pen_a[b2]
        k = ((r <= REV_K) | (ra <= REV_K)) & isq[si]
        rv.append(np.c_[si[k], pj[k] + st, np.minimum(r[k], cap), np.minimum(ra[k], cap)].astype(np.int32))
        if (st // 50_000) % 20 == 19:                  # progress every 1M records
            el, done = time.time() - t0, st + 50_000
            print(f"    reverse search: {done:,} / {len(pc):,} records, about "
                  f"{el / done * (len(pc) - done) / 60:.0f} min left", flush=True)
    del It1, It1a; gc.collect()
    rv = np.vstack(rv) if rv else np.zeros((0, 4), np.int32)          # int32: half the RAM
    rv = rv[np.argsort(rv[:, 0], kind="stable")]
    tick(t0, f"reverse search (name-led{' + address-led' if REV_ADDR_FETCH else ''}): {len(pc):,} pool records "
             f"-> {len(rv):,} pairs with queried S1")

    # ---- S1 side, in chunks of S1: penalised name / address searches + features (+ ranker)
    t0 = time.time()
    It = sp.vstack([X["nm"].B.T, X["ad"].B.T * np.float32(lam),
                    sp.csr_matrix(X["empty"].astype(np.float32)[None, :])], format="csr")
    n_out = 0
    for st in range(0, len(qrows), S1_CHUNK):
        rows = qrows[st:st + S1_CHUNK]
        An, Aa = X["nm"].prune(X["nm"].A[rows]), X["ad"].prune(X["ad"].A[rows])
        one = sp.csr_matrix(np.ones((len(rows), 1), np.float32))
        pairs = []
        for Q in (sp.hstack([An, Aa, one * np.float32(lam)], format="csr"),                 # name - lam*(1-addr)
                  sp.hstack([An * np.float32(mu), Aa * np.float32(1 / lam), one * 0], format="csr")):  # addr - mu*(1-name)
            qi, j, _ = top_k(Q, It, RETRIEVE_K, transposed=True)
            pairs.append(np.c_[rows[qi], j, np.full(len(j), REV_K + 1), np.full(len(j), REV_K + 1)])
        lo, hi = np.searchsorted(rv[:, 0], [rows.min(), rows.max() + 1])
        sel = rv[lo:hi]; sel = sel[np.isin(sel[:, 0], rows)]
        allp = np.vstack(pairs + [sel])
        d = pd.DataFrame(allp, columns=["i", "j", "r", "ra"]).groupby(["i", "j"], as_index=False)[["r", "ra"]].min()
        i, j = d.i.values.astype(np.int64), d.j.values.astype(np.int64)
        F = pair_features(i, j, d.r.values, d.ra.values, X)
        F.insert(0, "s1_id", pd.array(s1c.entity_id.values[i], dtype="string[pyarrow]"))   # compact strings
        F.insert(1, "pool_id", pd.array(pc.entity_id.values[j], dtype="string[pyarrow]"))
        F.insert(2, "country", country)
        if RANKER is not None:                       # keep only the final candidates (low RAM)
            F["p"] = RANKER.predict(F[FEATS])
            F["rank_p"] = rank_in_group(i, F.p.values)
            keep = select(F)
            F = F[keep]
        FETCHED.append(F); n_out += len(F)
        if (st // S1_CHUNK) % 10 == 9:                # progress every 10 chunks
            el, done = time.time() - t0, st + S1_CHUNK
            print(f"    S1-side search: {done:,} / {len(qrows):,} S1, about {el / done * (len(qrows) - done) / 60:.0f} "
                  f"min left", flush=True)
    del It; gc.collect()
    tick(t0, f"S1-side search + features -> {n_out:,} pairs ({n_out / len(qrows):.1f} per S1)")

    if TRUTH is not None:   # exact scores of every true pair (for the miss report)
        tr = TRUTH[TRUTH.s1_id.isin(set(s1c.entity_id.values[qrows]))]
        spos = pd.Series(np.arange(len(s1c)), index=s1c.entity_id.values)
        ppos = pd.Series(np.arange(len(pc)), index=pc.entity_id.values)
        ti, tj = spos.reindex(tr.s1_id).values, ppos.reindex(tr.pool_id).values
        ok = ~np.isnan(tj); ti, tj = ti[ok].astype(np.int64), tj[ok].astype(np.int64)
        TRUE_SCORES.append(pd.DataFrame({"s1_id": tr.s1_id.values[ok], "pool_id": tr.pool_id.values[ok],
                                         "n_exact": rowwise_dot(X["nm"].A, X["nm"].B, ti, tj),
                                         "a_exact": rowwise_dot(X["ad"].A, X["ad"].B, ti, tj)}))
    pd.concat(FETCHED[n_before:], ignore_index=True).to_parquet(ckpt + "_fetched.parquet", index=False)
    if TRUTH is not None:
        TRUE_SCORES[-1].to_parquet(ckpt + "_true.parquet", index=False)
    print(f"  {country} saved to checkpoint", flush=True)
    del X; gc.collect()

FETCHED = pd.concat(FETCHED, ignore_index=True)
TRUE_SCORES = pd.concat(TRUE_SCORES, ignore_index=True) if TRUE_SCORES else None
tick(T_ALL, f"all countries: {len(FETCHED):,} pairs")

## Stage 1 - Block 3b: stage-1 ranker (picks the final candidates)

In [ ]:
# ============================================================
# BLOCK 3b - ranker: picks the final candidates out of everything fetched
# ============================================================
# train: LightGBM (MIT) with 2-fold cross-validation by S1 -> every pair gets a score from a model
#        that never saw its S1 (honest recall), then one final model on all pairs is saved for test.
# test : the saved model was already applied inside BLOCK 3 (to save RAM) -> nothing to do here.
RANKER_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31, min_data_in_leaf=100,
                     lambda_l2=10.0, min_sum_hessian_in_leaf=1.0, feature_fraction=0.9, verbose=-1,
                     num_threads=N_THREADS, monotone_constraints=[MONO.get(f, 0) for f in FEATS])
RANKER_ROUNDS = 400

if LEARN:
    t0 = time.time()
    FETCHED["y"] = FETCHED.merge(TRUTH.assign(y=1), on=["s1_id", "pool_id"], how="left").y.fillna(0).values
    q = np.array(sorted(QUERY_IDS))
    fold = pd.Series(np.random.RandomState(SEED).randint(0, 2, len(q)), index=q).reindex(FETCHED.s1_id).values
    FETCHED["p"] = 0.0
    for k in (0, 1):
        tr, te = fold != k, fold == k
        m = lgb.train(RANKER_PARAMS, lgb.Dataset(FETCHED.loc[tr, FEATS], FETCHED.y[tr]), RANKER_ROUNDS)
        FETCHED.loc[te, "p"] = m.predict(FETCHED.loc[te, FEATS])
    FETCHED["rank_p"] = rank_in_group(pd.factorize(FETCHED.s1_id)[0], FETCHED.p.values)
    final = lgb.train(RANKER_PARAMS, lgb.Dataset(FETCHED[FEATS], FETCHED.y), RANKER_ROUNDS)
    final.save_model(f"{MODEL_DIR}/ranker.txt")
    imp = pd.Series(final.feature_importance("gain"), FEATS).sort_values(ascending=False)
    tick(t0, "ranker trained (2-fold scores + final model saved to learned/ranker.txt)")
    print("  most useful inputs:", ", ".join(imp.index[:8]))
CANDS = FETCHED[select(FETCHED)].reset_index(drop=True)
print(f"  final candidates: {len(CANDS):,} ({len(CANDS) / len(QUERY_IDS):.2f} per S1)")

## Stage 1 - Block 4: recall check

In [ ]:
# ============================================================
# BLOCK 4 - recall check against the ground truth (train only)
# ============================================================
def pct(x):
    return f"{100 * x:.2f}%"

if TRUTH is None:
    print("No ground truth for this split - skipping the recall check.")
else:
    s1_country = S1_CLEAN.set_index("entity_id").country
    T = TRUTH.merge(CANDS[["s1_id", "pool_id"]].assign(found=True), on=["s1_id", "pool_id"], how="left")
    T["found"] = T.found.fillna(False).astype(bool)
    T["fetched"] = T.merge(FETCHED[["s1_id", "pool_id"]].assign(f=1), how="left").f.notna().values
    T["country"] = s1_country.reindex(T.s1_id).values
    T["source"] = T.pool_id.str[:2]
    q = pd.Series(sorted(QUERY_IDS))
    per_s1 = CANDS.groupby("s1_id").size().reindex(q, fill_value=0)
    singles = ~q.isin(set(TRUTH.s1_id)).values
    print(f"PAIR RECALL        {pct(T.found.mean())}   ({T.found.sum():,} of {len(T):,} true pairs)")
    print(f"S1 fully covered   {pct(T.groupby('s1_id').found.all().mean())}   (every true match of the S1 is a candidate)")
    if LEARN:   # in 'train'/'test' only the final candidates are kept, so the ceiling equals recall
        print(f"ceiling            {pct(T.fetched.mean())}   (true pairs fetched before the ranker)")
    print(f"candidates per S1  mean {per_s1.mean():.2f} | median {per_s1.median():.0f} | "
          f"p95 {per_s1.quantile(.95):.0f} | max {per_s1.max()}")
    print(f"singletons         {singles.sum():,} | mean candidates {per_s1[singles].mean():.2f} | "
          f"with no candidate {pct((per_s1[singles] == 0).mean())}")
    print("\nrecall by country:\n" + T.groupby("country")[["found", "fetched"]].mean().round(4).to_string())
    print("\nrecall by source:\n" + T.groupby("source")[["found", "fetched"]].mean().round(4).to_string())
    rows = []
    for pmin in ([] if not LEARN else [0.01, 0.003, 0.001, 0.0003]):
        for topn in [5, 8, 10]:
            m = ((FETCHED.p >= pmin) | (FETCHED.rank_p <= topn) |
                 (KEEP_REV_BEST & ((FETCHED.r_rank == 1) | (FETCHED.ra_rank == 1))))
            rows.append({"P_MIN": pmin, "TOP_N": topn, "recall": round(FETCHED.y[m].sum() / len(T), 4),
                         "cands_per_s1": round(m.sum() / len(q), 2)})
    if rows:
      print("\nother ranker settings (same 2-fold scores):")
      print(pd.DataFrame(rows).pivot(index="P_MIN", columns="TOP_N", values=["recall", "cands_per_s1"]).to_string())

## Stage 1 - Block 5: missed true matches

In [ ]:
# ============================================================
# BLOCK 5 - false negatives: true matches NOT in the final candidates, and why
# ============================================================
#   "never fetched" : neither search direction brought it in -> look at the text / cleaning
#   "cut by ranker" : fetched, but the ranker scored it below P_MIN and outside TOP_N
_DOMAIN = re.compile(r"\.(?:com|net|org|co|in|fr|biz|info)\b|www\.", re.I)

if TRUTH is not None:
    M = T[~T.found].copy()
    M["reason"] = np.where(M.fetched, "cut by ranker", "never fetched")
    M = M.merge(FETCHED[["s1_id", "pool_id", "p", "rank_p", "r_rank", "ra_rank", "n", "a"]], how="left")
    M = M.merge(TRUE_SCORES, how="left")
    s1_raw = S1_ALL[S1_ALL.entity_id.isin(set(M.s1_id))].set_index("entity_id")
    p_raw = POOL[POOL.entity_id.isin(set(M.pool_id))].set_index("entity_id")
    s1_cl = S1_CLEAN[S1_CLEAN.entity_id.isin(set(M.s1_id))].set_index("entity_id")
    p_cl = POOL_CLEAN[POOL_CLEAN.entity_id.isin(set(M.pool_id))].set_index("entity_id")
    for side, raw, cl, key in [("s1", s1_raw, s1_cl, "s1_id"), ("p", p_raw, p_cl, "pool_id")]:
        M[f"{side}_name"] = raw.business_name.reindex(M[key]).values
        M[f"{side}_addr"] = raw.business_address.reindex(M[key]).values
        M[f"{side}_name_clean"] = cl.name_core.reindex(M[key]).values
        M[f"{side}_addr_clean"] = cl.address.reindex(M[key]).values
    M["p_addr_empty"] = M.p_addr_clean.fillna("").eq("")
    M["p_native_name"] = M.p_name.fillna("").str.contains(NATIVE_RE)
    M["p_domain_name"] = M.p_name.fillna("").str.contains(_DOMAIN)
    print(f"missed pairs: {len(M):,} of {len(T):,}\n")
    print("reason by country:\n" + pd.crosstab(M.country, M.reason, margins=True).to_string())
    pa = POOL_CLEAN[POOL_CLEAN.entity_id.isin(set(T.pool_id))].set_index("entity_id")
    pr = POOL[POOL.entity_id.isin(set(T.pool_id))].set_index("entity_id")
    as_bool = lambda s: s.reindex(T.pool_id).fillna(False).astype(bool).values
    base = pd.DataFrame({"p_addr_empty": as_bool(pa.address.eq("")),
                         "p_native_name": as_bool(pr.business_name.str.contains(NATIVE_RE)),
                         "p_domain_name": as_bool(pr.business_name.str.contains(_DOMAIN))})
    trait = pd.DataFrame({"share_in_misses": M[base.columns].mean(), "share_in_all_true_pairs": base.mean()})
    trait["times_more_likely_missed"] = trait.share_in_misses / trait.share_in_all_true_pairs
    print("\ntraits:\n" + trait.round(3).to_string())

    def show_misses(reason=None, country=None, n=15, seed=0, clean=True):
        """reason: 'never fetched' | 'cut by ranker'"""
        d = M
        if reason: d = d[d.reason == reason]
        if country: d = d[d.country == country]
        d = d.sample(min(n, len(d)), random_state=seed)
        for _, r in d.iterrows():
            extra = (f"p {r.p:.1e}, rank {r.rank_p:.0f}, reverse rank name-led {r.r_rank:.0f} / address-led "
                     f"{r.ra_rank:.0f}" if r.reason == "cut by ranker"
                     else "not fetched")
            print(f"[{r.reason} | name {r.n_exact:.2f}, address {r.a_exact:.2f} | {extra}]")
            print(f"   S1 : {r.s1_name}  |  {r.s1_addr}")
            print(f"   {r.pool_id[:2]} : {r.p_name}  |  {r.p_addr}")
            if clean:
                print(f"   clean S1 : {r.s1_name_clean}  |  {r.s1_addr_clean}")
                print(f"   clean {r.pool_id[:2]} : {r.p_name_clean}  |  {r.p_addr_clean}")

    def show_s1(s1_id, n=10):
        """One S1: its true matches (found or not) and its best candidates by ranker score."""
        print("S1:", S1_ALL[S1_ALL.entity_id == s1_id][["business_name", "business_address"]].values.tolist())
        print("true matches:\n" + T[T.s1_id == s1_id][["pool_id", "found", "fetched"]].to_string(index=False))
        cs = FETCHED[FETCHED.s1_id == s1_id].sort_values("p", ascending=False).head(n)
        raw = POOL[POOL.entity_id.isin(set(cs.pool_id))].set_index("entity_id")
        cs = cs.assign(name=raw.business_name.reindex(cs.pool_id).values,
                       address=raw.business_address.reindex(cs.pool_id).values, true=cs.y == 1)
        print("top fetched:\n" + cs[["pool_id", "true", "p", "n", "a", "r_rank", "name", "address"]].to_string(index=False))

    show_misses(n=10)
    # more: show_misses('never fetched', 'india', n=20) | show_s1(M.s1_id.iloc[0])
    # M.to_csv(f"{WORK_DIR}/misses.csv", index=False)   # to look at them in a sheet

## Stage 1 - Block 6: save stage-1 outputs

In [ ]:
# ============================================================
# BLOCK 6 - save the outputs
# ============================================================
#  candidate_pairs_<tag>.tsv       submission format, one row per queried S1 (empty = no candidate)
#  candidate_pairs.tsv             the file for the final package (full test run only)
#  candidates_<tag>.parquet        every final candidate pair + its features + ranker score p (stage 2 input)
#  final_clean_<tag>_{s1,pool}.parquet   cleaned + raw text of the records involved (stage 2 input)
tag = f"{SPLIT}_n{N_S1}_pf{POOL_FRACTION}"
lists = CANDS.groupby("s1_id").pool_id.agg(",".join)
out = pd.DataFrame({"source1_entity_id": S1_ALL.entity_id[S1_ALL.entity_id.isin(QUERY_IDS)].values})
out["candidate_entity_ids"] = out.source1_entity_id.map(lists).fillna("")
out.to_csv(f"{WORK_DIR}/candidate_pairs_{tag}.tsv", sep="\t", index=False)
if SPLIT == "test" and N_S1 is None and POOL_FRACTION == 1.0 and COUNTRIES is None:
    out.to_csv(f"{WORK_DIR}/candidate_pairs.tsv", sep="\t", index=False)
    print("  wrote candidate_pairs.tsv (final submission file)")
CANDS.drop(columns=[c for c in ["y"] if c in CANDS]).to_parquet(f"{WORK_DIR}/candidates_{tag}.parquet", index=False)

cols = ["entity_id", "country", "name", "name_core", "address", "nat_name", "nat_addr"]
m1 = S1_CLEAN.entity_id.isin(QUERY_IDS).values
m2 = POOL_CLEAN.entity_id.isin(set(CANDS.pool_id)).values
S1_CLEAN.loc[m1, cols].assign(raw_name=S1_ALL.business_name.values[m1], raw_address=S1_ALL.business_address.values[m1]) \
    .to_parquet(f"{WORK_DIR}/final_clean_{tag}_s1.parquet", index=False)
POOL_CLEAN.loc[m2, cols].assign(raw_name=POOL.business_name.values[m2], raw_address=POOL.business_address.values[m2]) \
    .to_parquet(f"{WORK_DIR}/final_clean_{tag}_pool.parquet", index=False)
print(f"saved candidate_pairs_{tag}.tsv ({len(out):,} rows, {(out.candidate_entity_ids == '').mean():.2%} with no "
      f"candidate), candidates_{tag}.parquet and final_clean_{tag}_*.parquet  -> inputs for stage 2")

## Stage 2 - Block 1: settings + load stage-1 outputs

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 1: settings + load the stage-1 outputs
# ============================================================
# runs right after stage 1 in the same session ('learn' / 'train' -> trains, 'test' -> applies)
# !pip -q install xgboost          # only if you add "xgb" to MODELS
from rapidfuzz.distance import JaroWinkler, Levenshtein
MODE      = 'test' if STEP == 'test' else 'train'
TAG       = f"{SPLIT}_n{N_S1}_pf{POOL_FRACTION}"      # the stage-1 run whose candidates are used
OUT_DIR   = f'{WORK_ROOT}/stage2'
GT_PATH   = f'{DATA_ROOT}/train/train_ground_truth.tsv'
os.makedirs(OUT_DIR, exist_ok=True)
N_FOLDS   = 5            # K-fold by S1 = K rotating 80/20 splits: every S1 is in the 20% exactly once
N_REPEATS = 1            # repeat the K-fold with another random split (2-3 gives a steadier average)
MODELS    = ["lgb"]      # ["lgb", "xgb"] = average of LightGBM and XGBoost (MIT / Apache). Tested: they
                         #   make the same errors (91% shared) -> no gain, so off by default
ENSEMBLE  = "folds"      # "folds": test scores = average of the N_FOLDS x N_REPEATS models, each trained on
                         #   a different 80% | "full": one extra model trained on 100% (tested: same F0.5)
LGB_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_data_in_leaf=50, lambda_l2=5.0,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbose=-1,
                  num_threads=N_THREADS, seed=SEED)
XGB_PARAMS = dict(objective="binary:logistic", eta=0.05, max_depth=7, min_child_weight=5, subsample=0.8,
                  colsample_bytree=0.8, reg_lambda=5.0, nthread=N_THREADS, seed=SEED)
ROUNDS    = 600
SCORE_CHUNK = 2_000_000  # test: pairs scored per chunk (lower it if RAM runs out)

# free stage-1 memory, then load its saved outputs
for _v in ["S1_ALL", "POOL", "S1_CLEAN", "POOL_CLEAN", "FETCHED", "CANDS", "TRUE_SCORES", "M", "T"]:
    globals().pop(_v, None)
gc.collect()
t0 = time.time()
C = pd.read_parquet(f"{WORK_DIR}/candidates_{TAG}.parquet").rename(columns={"p": "p1", "rank_p": "rank_p1"})
S1T = pd.read_parquet(f"{WORK_DIR}/final_clean_{TAG}_s1.parquet")
POOLT = pd.read_parquet(f"{WORK_DIR}/final_clean_{TAG}_pool.parquet")
ALL_S1 = S1T.entity_id.values                     # every queried S1 (also those with no candidate)
C = C.sort_values("s1_id", kind="stable").reset_index(drop=True)
if MODE == "train":
    TRUTH = truth_pairs(pd.read_csv(GT_PATH, sep="\t", dtype=str, keep_default_na=False))
    TRUTH = TRUTH[TRUTH.s1_id.isin(set(ALL_S1))].reset_index(drop=True)
    C["y"] = C.merge(TRUTH.assign(y=1), how="left").y.fillna(0).astype(np.int8).values
    print(f"  true pairs {len(TRUTH):,} | in candidates {int(C.y.sum()):,} (stage-1 recall "
          f"{C.y.sum() / len(TRUTH):.4f}) | singletons {len(set(ALL_S1) - set(TRUTH.s1_id)):,}")
tick(t0, f"stage 2 input: {len(C):,} candidate pairs for {len(ALL_S1):,} S1 ({len(C) / len(ALL_S1):.2f} per S1)")

## Stage 2 - Block 2: pair features

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 2: pair features
# ============================================================
# stage-1 features (TF-IDF scores, ranks from both search directions, house numbers, words, legal
# conflict, stage-1 ranker score) + string similarities + HOW the two records differ:
#   house numbers: same / cut off (600 vs 60) / one digit changed / two digits swapped / other
#   name & address words: extra words that look like typos vs different real words (a look-alike
#   business usually swaps a real word, "Amis" -> "Amicale"; noise usually makes typos or drops words)
_DOMAIN = re.compile(r"\.(?:com|net|org|co|in|fr|biz|info)\b|www\.", re.I)
_POSTAL = re.compile(r"^\d{5,6}$")
STAGE1_FEATS = ["n", "a", "empty", "legal", "pen_n", "pen_a", "r_rank", "pool_best", "margin", "rank_n", "rank_a",
                "ra_rank", "pool_best_a", "margin_a", "num_j", "num_first", "num_sub", "num_extra",
                "w_share_p", "w_share_s", "first_eq", "nw_s", "nw_p", "p1", "rank_p1"]

def _pw(a, b, scorer, scale=100.0):
    return process.cpdist(a, b, scorer=scorer, workers=-1, dtype=np.float32) / scale

def _num_rel(x, y):
    """0 same | 1 cut off (60/600, 1501/150) | 2 one digit changed | 3 two digits swapped | 4 other"""
    if x == y: return 0
    if x.startswith(y) or y.startswith(x) or x.endswith(y) or y.endswith(x): return 1
    if len(x) == len(y):
        d = [k for k in range(len(x)) if x[k] != y[k]]
        if len(d) == 1: return 2
        if len(d) == 2 and x[d[0]] == y[d[1]] and x[d[1]] == y[d[0]]: return 3
    return 2 if Levenshtein.distance(x, y) == 1 else 4

def _number_feats(x, y):
    X = [w for w in x.split() if w.isdigit()]; Y = [w for w in y.split() if w.isdigit()]
    if not X or not Y:
        return (-1, -1, -1, -1, -1, -1)
    best = [min(_num_rel(u, v) for u in X) for v in Y]          # best explanation of each S2/S3 number
    try:
        delta = float(np.log1p(abs(int(X[0][:9]) - int(Y[0][:9]))))
    except ValueError:
        delta = -1.0
    return (_num_rel(X[0], Y[0]), sum(b == 4 for b in best), sum(b in (2, 3) for b in best),
            sum(b == 1 for b in best), delta, min(best))

def _word_feats(x, y, voc, min_n):
    A, B = x.split(), y.split(); sa, sb = set(A), set(B)
    extra = [w for w in B if w not in sa]; miss = [w for w in A if w not in sb]
    close = lambda w, other: max((fuzz.ratio(w, o) for o in other), default=0)
    ex = [close(w, A) for w in extra]
    return (len(extra), sum(s < 70 and voc.get(w, 0) >= min_n and len(w) >= 3 for w, s in zip(extra, ex)),
            sum(s >= 70 for s in ex), sum(close(w, B) < 70 for w in miss), min(ex) if ex else 100, len(miss))

def make_features(C, S1T, POOLT, voc, min_n):
    s = S1T.set_index("entity_id").reindex(C.s1_id)
    p = POOLT.set_index("entity_id").reindex(C.pool_id)
    n1, n2 = s.name_core.fillna("").tolist(), p.name_core.fillna("").tolist()
    a1, a2 = s.address.fillna("").tolist(), p.address.fillna("").tolist()
    F = pd.DataFrame(index=C.index)
    F["n_set"], F["n_sort"] = _pw(n1, n2, fuzz.token_set_ratio), _pw(n1, n2, fuzz.token_sort_ratio)
    F["n_part"], F["n_ratio"] = _pw(n1, n2, fuzz.partial_ratio), _pw(n1, n2, fuzz.ratio)
    j1, j2 = [x.replace(" ", "") for x in n1], [x.replace(" ", "") for x in n2]
    F["n_jw"] = _pw(j1, j2, JaroWinkler.normalized_similarity, 1.0)
    F["n_glued_eq"] = np.array([x == y and x != "" for x, y in zip(j1, j2)], np.float32)
    k1 = [" ".join(skeleton(w) for w in x.split()) for x in n1]
    k2 = [" ".join(skeleton(w) for w in x.split()) for x in n2]
    F["n_sound"] = _pw(k1, k2, fuzz.token_set_ratio)
    F["n_prefix"] = np.array([bool(y) and x.split()[:len(y.split())] == y.split() for x, y in zip(n1, n2)], np.float32)
    F["n_full_ratio"] = _pw(s.name.fillna("").tolist(), p.name.fillna("").tolist(), fuzz.ratio)
    F["a_set"], F["a_part"] = _pw(a1, a2, fuzz.token_set_ratio), _pw(a1, a2, fuzz.partial_ratio)
    F["a_ratio"] = _pw(a1, a2, fuzz.ratio)
    pc = []
    for x, y in zip(a1, a2):
        px = {t for t in x.split() if _POSTAL.match(t)}; py = {t for t in y.split() if _POSTAL.match(t)}
        pc.append(-1.0 if not px or not py else float(bool(px & py)))
    F["postal_eq"] = np.array(pc, np.float32)
    nf = np.array([_number_feats(x, y) for x, y in zip(a1, a2)], np.float32)
    for k, c in enumerate(["num_rel_first", "num_unexplained", "num_typo", "num_cut", "num_delta", "num_best_rel"]):
        F[c] = nf[:, k]
    wn = np.array([_word_feats(x, y, voc, min_n) for x, y in zip(n1, n2)], np.float32)
    strip = lambda t: " ".join(w for w in t.split() if not w.isdigit())
    wa = np.array([_word_feats(strip(x), strip(y), voc, min_n) for x, y in zip(a1, a2)], np.float32)
    for k, c in enumerate(["extra", "substituted", "typo", "dropped", "extra_minsim", "missing"]):
        F[f"nw_{c}"], F[f"aw_{c}"] = wn[:, k], wa[:, k]
    F["p_native"] = p.nat_name.fillna(False).astype(np.float32).values
    F["p_domain"] = p.raw_name.fillna("").str.contains(_DOMAIN).astype(np.float32).values
    F["p_s3"] = C.pool_id.str.startswith("S3").astype(np.float32).values
    F["len_ratio"] = (np.array([len(y) for y in j2], np.float32) + 1) / (np.array([len(x) for x in j1], np.float32) + 1)
    g = pd.factorize(C.s1_id)[0]                      # how the pair compares with the S1's other candidates
    F["s1_n_cands"] = np.bincount(g)[g].astype(np.float32)
    F["s1_max_p1"] = C.groupby("s1_id").p1.transform("max").values.astype(np.float32)
    F["p1_gap"] = F.s1_max_p1 - C.p1.values
    F["s1_n_p1_hi"] = (C.p1 > 0.5).groupby(C.s1_id).transform("sum").values.astype(np.float32)
    return pd.concat([C[STAGE1_FEATS].astype(np.float32), F], axis=1)

# word counts over the business names seen here ("is this a real word?"), relative to data size
VOC = collections.Counter(w for x in pd.concat([S1T.name_core, POOLT.name_core]) for w in set(x.split()))
MIN_N = max(3, int(2e-5 * (len(S1T) + len(POOLT))))
if MODE == "train":
    t0 = time.time()
    X = make_features(C, S1T, POOLT, VOC, MIN_N)
    FEATURES = list(X.columns)
    tick(t0, f"{len(FEATURES)} features for {len(C):,} pairs")

## Stage 2 - Block 3: picking rules + F0.5

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 4: from scores to match lists
# ============================================================
# one owner per S2/S3 record (each belongs to at most one S1), then per S1 a threshold or the
# "expected F0.5" rule (keep the top-m candidates that maximise the expected F0.5; m = 0 = no match)
def one_owner(c, col):
    return c[c[col].values >= c.groupby("pool_id")[col].transform("max").values]

def pick_threshold(c, col, t):
    return one_owner(c[c[col] >= t], col)

def pick_expected_f(c, col, t_min=0.05, beta2=0.25):
    c = one_owner(c[c[col] >= t_min], col).sort_values(["s1_id", col], ascending=[True, False])
    p = c[col].values.astype(np.float64)
    g = pd.factorize(c.s1_id.values)[0]
    m = c.groupby(g).cumcount().values + 1
    cum = pd.Series(p).groupby(g).cumsum().values
    tot = np.bincount(g, weights=p)[g]
    ef = (1 + beta2) * cum / (beta2 * m + tot)
    ef0 = np.exp(np.bincount(g, weights=np.log1p(-np.minimum(p, 1 - 1e-9))))[g]
    best = pd.Series(ef).groupby(g).transform("max").values
    best_m = pd.Series(np.where(ef >= best, m, 10 ** 9)).groupby(g).transform("min").values
    return c[(m <= best_m) & (best > ef0)]

def apply_rule(c, rule):
    return pick_threshold(c, "p", rule["t"]) if rule["method"] == "threshold" else pick_expected_f(c, "p", rule["t_min"])

def macro_f05(pred, truth, s1_ids, beta2=0.25):
    """the challenge metric: F0.5 per S1, averaged; a singleton scores 1 only if left empty"""
    idx = pd.Index(pd.unique(np.asarray(s1_ids))); n = len(idx)
    count = lambda d: np.bincount(idx.get_indexer(d.s1_id.values), minlength=n)
    tp, npred, ntrue = count(pred.merge(truth[["s1_id", "pool_id"]])), count(pred), count(truth)
    with np.errstate(divide="ignore", invalid="ignore"):
        pr, rc = tp / npred, tp / ntrue
        f = np.where(tp > 0, (1 + beta2) * pr * rc / (beta2 * pr + rc), 0.0)
    f = np.where(ntrue == 0, (npred == 0).astype(float), f)
    return f.mean(), pd.Series(f, index=idx)

## Stage 2 - Block 4: matching model + K-fold F0.5

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 4: matching model + honest K-fold F0.5
# ============================================================
# K-fold by S1 = K rotating 80/20 splits. Each held-out 20% is scored by a model that never saw
# those S1, and its threshold is chosen on the other 80% only -> F0.5 per fold, then mean +- std.
# The K (x repeats) fold models are kept: with ENSEMBLE="folds" the test score is their average.
def fit_model(Xtr, ytr):
    models = {}
    if "lgb" in MODELS:
        models["lgb"] = lgb.train(LGB_PARAMS, lgb.Dataset(Xtr, ytr), ROUNDS)
    if "xgb" in MODELS:
        import xgboost as xgb
        models["xgb"] = xgb.train(XGB_PARAMS, xgb.DMatrix(Xtr, ytr), ROUNDS)
    return models

def predict(models, Xa):
    ps = []
    for name, m in models.items():
        if name == "lgb":
            ps.append(m.predict(Xa))
        else:
            import xgboost as xgb
            ps.append(m.predict(xgb.DMatrix(Xa)))
    return np.mean(ps, axis=0)

def save_models(models, suffix):
    files = []
    for name, m in models.items():
        f = f"{OUT_DIR}/matcher_{name}_{suffix}.{'txt' if name == 'lgb' else 'json'}"
        m.save_model(f); files.append(os.path.basename(f))
    return files

def load_model(fname):
    if fname.startswith("matcher_lgb"):
        return {"lgb": lgb.Booster(model_file=f"{OUT_DIR}/{fname}")}
    import xgboost as xgb
    b = xgb.Booster(); b.load_model(f"{OUT_DIR}/{fname}")
    return {"xgb": b}

# ---- fast F0.5 for threshold rules (array maths instead of merges: ~100x faster)
S1_INDEX = pd.Index(ALL_S1)
G = S1_INDEX.get_indexer(C.s1_id)                    # S1 number of each pair
_PC = pd.factorize(C.pool_id)[0]

def owner_mask(p):
    """True where this S1 is the S2/S3 record's highest-scoring S1 (one owner per record)"""
    mx = np.full(_PC.max() + 1 if len(_PC) else 0, -np.inf); np.maximum.at(mx, _PC, p)
    return p >= mx[_PC]

def f05_per_s1(pred, y, ntrue, beta2=0.25):
    tp = np.bincount(G, weights=(pred & y).astype(float), minlength=len(S1_INDEX))
    npred = np.bincount(G, weights=pred.astype(float), minlength=len(S1_INDEX))
    with np.errstate(divide="ignore", invalid="ignore"):
        pr, rc = tp / npred, tp / ntrue
        f = np.where(tp > 0, (1 + beta2) * pr * rc / (beta2 * pr + rc), 0.0)
    return np.where(ntrue == 0, (npred == 0).astype(float), f)

THRESHOLDS = np.round(np.arange(0.3, 0.96, 0.05), 2)

if MODE == "train":
    t0 = time.time()
    y = C.y.values.astype(bool)
    NTRUE = np.bincount(S1_INDEX.get_indexer(TRUTH.s1_id), minlength=len(S1_INDEX)).astype(float)
    C["p"] = 0.0
    FOLD_SCORES, MODEL_FILES = [], []
    for rep in range(N_REPEATS):
        fold_all = np.random.RandomState(SEED + rep).randint(0, N_FOLDS, len(S1_INDEX))  # every queried S1
        fold = fold_all[G]
        oof = np.zeros(len(C))
        for k in range(N_FOLDS):                      # train on 80%, score the other 20%
            tr, te = fold != k, fold == k
            models = fit_model(X[tr], y[tr])
            oof[te] = predict(models, X[te])
            if ENSEMBLE == "folds":
                MODEL_FILES += save_models(models, f"r{rep}_f{k}")
        C["p"] += oof / N_REPEATS
        own = owner_mask(oof)
        per_t = {t: f05_per_s1(own & (oof >= t), y, NTRUE) for t in THRESHOLDS}
        for k in range(N_FOLDS):                      # threshold chosen on the other 80% only
            t_best = max(THRESHOLDS, key=lambda t: per_t[t][fold_all != k].mean())
            FOLD_SCORES.append(per_t[t_best][fold_all == k].mean())
    FOLD_SCORES = np.array(FOLD_SCORES)
    tick(t0, f"{N_REPEATS} x {N_FOLDS}-fold done")
    print(f"  F0.5 per held-out 20%: {np.round(FOLD_SCORES, 4).tolist()}")
    print(f"  F0.5 mean {FOLD_SCORES.mean():.4f} +- {FOLD_SCORES.std():.4f}")
    if ENSEMBLE == "full":
        t0 = time.time()
        MODEL_FILES = save_models(fit_model(X, y), "full")
        tick(t0, "final model trained on 100% and saved")
    json.dump({"features": FEATURES, "model_files": MODEL_FILES}, open(f"{OUT_DIR}/matcher_features.json", "w"))
    m0 = load_model(MODEL_FILES[0])
    if "lgb" in m0:
        imp = pd.Series(m0["lgb"].feature_importance("gain"), FEATURES).sort_values(ascending=False)
        print("  most useful inputs:", ", ".join(imp.index[:12]))
    print(f"  saved {len(MODEL_FILES)} model file(s): {', '.join(MODEL_FILES[:3])}{' ...' if len(MODEL_FILES) > 3 else ''}")
else:                                                # test: score in chunks (keeps RAM low)
    t0 = time.time()
    meta = json.load(open(f"{OUT_DIR}/matcher_features.json"))
    MATCHERS = [load_model(f) for f in meta["model_files"]]
    p = np.zeros(len(C))
    bounds = sorted(set(np.searchsorted(C.s1_id.values, C.s1_id.values[::SCORE_CHUNK]).tolist() + [len(C)]))
    for a, b in zip(bounds[:-1], bounds[1:]):        # chunks never split an S1 (C is sorted by s1_id)
        Xc = make_features(C.iloc[a:b], S1T, POOLT, VOC, MIN_N)[meta["features"]]
        p[a:b] = np.mean([predict(m, Xc) for m in MATCHERS], axis=0)
    C["p"] = p
    tick(t0, f"scored {len(C):,} test pairs with {len(MATCHERS)} model(s) averaged")

## Stage 2 - Block 5: choose the rule, final report

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 5: choose the picking rule on the cross-validated scores, final report
# ============================================================
if MODE == "train":
    p_all = C.p.values
    own = owner_mask(p_all)
    res = [{"method": "threshold", "t": float(t), "F05": f05_per_s1(own & (p_all >= t), y, NTRUE).mean()}
           for t in THRESHOLDS]
    for t_min in (0.02, 0.05, 0.1, 0.2):              # expected-F0.5 rule (slower, only 4 settings)
        res.append({"method": "expected_f", "t_min": t_min,
                    "F05": macro_f05(apply_rule(C, {"method": "expected_f", "t_min": t_min}), TRUTH, ALL_S1)[0]})
    res = pd.DataFrame(res).sort_values("F05", ascending=False)
    RULE = {k: v for k, v in res.iloc[0].to_dict().items() if k in ("method", "t", "t_min") and pd.notna(v)}
    json.dump(RULE, open(f"{OUT_DIR}/matcher_rule.json", "w"))
    print(res.head(6).to_string(index=False))
    PRED = apply_rule(C, RULE)
    f, per = macro_f05(PRED, TRUTH, ALL_S1)
    sing = ~per.index.isin(set(TRUTH.s1_id))
    tp = len(PRED.merge(TRUTH))
    print(f"\nMACRO F0.5 {f:.4f} with rule {RULE}   (per-fold mean {FOLD_SCORES.mean():.4f} +- {FOLD_SCORES.std():.4f})")
    print(f"  singletons {per[sing].mean():.4f} ({sing.sum():,}) | S1 with matches {per[~sing].mean():.4f} ({(~sing).sum():,})")
    print(f"  pair precision {tp / max(len(PRED), 1):.4f} | pair recall {tp / len(TRUTH):.4f}")
    ctry = S1T.set_index("entity_id").country
    print("  by country: " + ", ".join(f"{k} {v:.4f}" for k, v in per.groupby(ctry.reindex(per.index).values).mean().items()))
    print(f"  best possible with these candidates: {macro_f05(C[C.y == 1], TRUTH, ALL_S1)[0]:.4f}")
else:
    RULE = json.load(open(f"{OUT_DIR}/matcher_rule.json"))
    PRED = apply_rule(C, RULE)

## Stage 2 - Block 6: wrong merges and missed matches

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 6: wrong merges and missed matches (train)
# ============================================================
if MODE == "train":
    FP = PRED.merge(TRUTH.assign(t=1), how="left"); FP = FP[FP.t.isna()]
    FN = C[C.y == 1].merge(PRED[["s1_id", "pool_id"]].assign(k=1), how="left"); FN = FN[FN.k.isna()]
    print(f"wrong merges: {len(FP):,} ({FP.s1_id.isin(set(ALL_S1) - set(TRUTH.s1_id)).sum():,} on singletons) | "
          f"missed matches in the candidates: {len(FN):,} | not in the candidates (stage 1): {len(TRUTH) - int(C.y.sum()):,}")
    s_txt, p_txt = S1T.set_index("entity_id"), POOLT.set_index("entity_id")

    def show_pairs(d, n=8, seed=0):
        for _, r in d.sample(min(n, len(d)), random_state=seed).iterrows():
            print(f"[p {r.p:.3f} | name {r.n:.2f} addr {r.a:.2f} | stage-1 p {r.p1:.3f}]")
            print(f"   S1 : {s_txt.raw_name.get(r.s1_id)} | {s_txt.raw_address.get(r.s1_id)}")
            print(f"   {r.pool_id[:2]} : {p_txt.raw_name.get(r.pool_id)} | {p_txt.raw_address.get(r.pool_id)}")
    print("\n--- wrong merges"); show_pairs(FP)
    print("\n--- missed matches"); show_pairs(FN)

## Stage 2 - Block 7: submission files + checks (test)

In [ ]:
# ============================================================
# STAGE 2 - BLOCK 7: write matching_results.tsv + candidate_pairs.tsv and check them (test)
# ============================================================
def write_lists(pairs, s1_ids, col, path):
    lists = pairs.groupby("s1_id").pool_id.agg(lambda s: ",".join(pd.unique(s)))
    out = pd.DataFrame({"source1_entity_id": s1_ids})
    out[col] = out.source1_entity_id.map(lists).fillna("")
    out.to_csv(path, sep="\t", index=False)
    return out

def check_submission(match, cand, test_dir):
    """the challenge rules: one row per test S1, only existing S2/S3 ids, no duplicates,
    matches must be a subset of the candidates"""
    s1 = pd.read_csv(f"{test_dir}/test_source1.tsv", sep="\t", dtype=str, usecols=["entity_id"]).entity_id
    pool = set(pd.concat([pd.read_csv(f"{test_dir}/test_source{k}.tsv", sep="\t", dtype=str,
                                      usecols=["entity_id"]).entity_id for k in (2, 3)]))
    issues = []
    for name, df, col in [("matching", match, "matched_entity_ids"), ("candidate", cand, "candidate_entity_ids")]:
        if df.source1_entity_id.duplicated().any(): issues.append(f"{name}: duplicate S1 rows")
        if set(df.source1_entity_id) != set(s1): issues.append(f"{name}: S1 rows differ from test_source1")
        ids = df[col].str.split(",").explode(); ids = ids[ids.notna() & (ids != "")]
        if not ids.str.match(r"^S[23]-").all(): issues.append(f"{name}: non S2/S3 id")
        if not set(ids) <= pool: issues.append(f"{name}: id not in the test S2/S3 files")
        if df[col].str.split(",").map(lambda l: len(l) != len(set(l))).any(): issues.append(f"{name}: duplicate id in a list")
    m = match.assign(i=match.matched_entity_ids.str.split(",")).explode("i")
    c = cand.assign(i=cand.candidate_entity_ids.str.split(",")).explode("i")
    m = m[m.i != ""]; c = c[c.i != ""]
    if len(m.merge(c, on=["source1_entity_id", "i"])) != len(m): issues.append("a match is not in the candidates")
    print("CHECK:", "PASS" if not issues else "\n  ".join(["FAIL"] + issues))

if MODE == "test":
    os.makedirs(f"{OUT_DIR}/output", exist_ok=True)
    match = write_lists(PRED, ALL_S1, "matched_entity_ids", f"{OUT_DIR}/output/matching_results.tsv")
    cand = write_lists(C, ALL_S1, "candidate_entity_ids", f"{OUT_DIR}/output/candidate_pairs.tsv")   # = model input
    print(f"matching_results.tsv: {len(match):,} rows | {(match.matched_entity_ids == '').mean():.2%} predicted "
          f"'no match' | {len(PRED):,} matched pairs ({len(PRED) / len(match):.2f} per S1)")
    print(f"candidate_pairs.tsv : {len(C):,} candidate pairs ({len(C) / len(match):.2f} per S1)")
    check_submission(match, cand, f"{DATA_ROOT}/test")